이 문서는 hwpapi가 지원하는 기능을 **카테고리별로** 소개합니다. 여기에 나오는 모든 코드는 `tests/smoke_scenarios.py` 와 `tests/smoke_features.py` 에서 실제로 HWP를 띄워놓고 검증한 것입니다.

::: {.callout-tip}
문서를 따라가기 전에 `from hwpapi.core import App; app = App()` 으로 HWP를 띄워놓고 실습하면 이해가 빠릅니다.
:::

## 1. 문자 서식 (CharShape)

### 1.1 기본 서식

```python
app.insert_text("BOLD ITALIC UNDERLINE")
app.select_text()  # 현재 줄 선택
app.set_charshape(
    bold=True,
    italic=True,
    underline_type=1,   # 1=single
)
```

적용된 서식 **읽어오기**:

```python
cs = app.get_charshape()
# get_charshape는 마지막 적용 값의 캐시를 반환
# 현재 커서 위치의 실제 값을 원하면 GetDefault로 refresh:
cs_action = app.actions.CharShape
cs_action.act.GetDefault(cs_action.pset._raw)
print(cs_action.pset.Bold)       # 1
print(cs_action.pset.Italic)     # 1
print(cs_action.pset.UnderlineType)  # 1
```

### 1.2 색상

색은 **hex 문자열**로 입력합니다. 내부적으로 HWP의 BBGGRR 포맷으로 변환됩니다.

```python
app.set_charshape(
    text_color="#FF0000",   # 빨강
    shade_color="#FFFF00",  # 형광펜 노랑
)
```

### 1.3 크기 & 자간

```python
app.set_charshape(
    height=2400,        # 24pt (HWPUNIT, 100=1pt)
    spacing_hangul=50,  # 한글 자간 (%)
)
```

### 1.4 장식 (위/아래첨자, 취소선)

```python
# H₂O의 '2'만 선택해서 아래첨자 적용
app.set_charshape(sub_script=1)

# X²의 '2'만 선택해서 위첨자
app.set_charshape(super_script=1)

# 취소선
app.set_charshape(strike_out_type=1)
```

## 2. 문단 서식 (ParaShape)

```python
app.set_parashape(
    align_type=3,       # 1=left, 2=right, 3=center, 4=justify
    line_spacing=200,   # 200% 줄간격
    indentation=2000,   # 첫줄 들여쓰기 (HWPUNIT)
)

pa = app.get_parashape()
print(pa.AlignType, pa.LineSpacing, pa.Indentation)
```

## 3. 텍스트 조작

### 3.1 삽입·삭제·탭

```python
app.insert_text("새 텍스트\n")
app.actions.InsertTab.run()          # 탭 문자
app.actions.SelectAll.run()
app.api.Run("Delete")                # 모두 삭제
```

### 3.2 유니코드 특수문자

한글, 한자, 기호 모두 그대로 입력 가능합니다.

```python
app.insert_text("© ® ™ → ★ 数 ⚡")
```

### 3.3 찾기·바꾸기

```python
app.find_text("검색어")                      # 이동
count = app.replace_all("before", "after")   # 치환 개수 반환

# RepeatFind로 연속 검색
app.move.top_of_file()
app.find_text("Apple")
for _ in range(3):
    app.actions.RepeatFind.run()
```

### 3.4 서식 조건부 검색

```python
action = app.actions.ForwardFind
action.pset.FindString = "중요"
action.pset.find_char_shape.bold = True   # 굵은 글씨만 찾기
action.run()
```

### 3.5 복사 · 붙여넣기

```python
app.actions.SelectAll.run()
app.actions.Copy.run()
app.move.bottom_of_file()
app.actions.Paste.run()
```

### 3.6 Undo · Redo

```python
app.insert_text("새로운 문장")
app.actions.Undo.run()   # 취소
app.actions.Redo.run()   # 다시
```

## 4. 표

```python
# 3x4 표 생성
action = app.actions.TableCreate
action.pset.Rows = 3
action.pset.Cols = 4
action.run()

# 셀 채우기
for i in range(12):
    app.insert_text(f"C{i + 1}")
    app.api.Run("TableRightCell")
```

## 5. 탐색 (Move)

```python
app.move.top_of_file()       # 문서 처음으로
app.move.bottom_of_file()    # 문서 끝으로
app.move.current_list(para=5, pos=0)  # 5번째 문단 시작

# 줄바꿈 기준 이동
app.api.Run("MoveLineBegin")
app.api.Run("MoveLineEnd")
app.api.Run("MoveLineUp")
app.api.Run("MoveLineDown")
```

## 6. 단위 변환 (UnitProperty)

`UnitProperty` 를 갖는 속성은 익숙한 단위를 그대로 쓸 수 있습니다.

```python
from hwpapi.parametersets import PageDef

page = PageDef()
page.width = "210mm"    # A4 너비
page.width = "21cm"     # 동일
page.width = "8.27in"   # 동일
page.width = 210        # bare number, 기본 단위(mm)

# 읽기는 기본 단위(mm)로
print(page.width)  # 210.0
```

**지원 단위:**

| 단위 | HWPUNIT 환산 | 주용도 |
|------|-------------|--------|
| `mm` | ×283        | 페이지 크기, 여백 |
| `cm` | ×2830       | 페이지 크기 |
| `in` | ×7200       | 영문 레터지 |
| `pt` | ×100        | 글자 크기, 여백 |

## 7. 파일 입출력

```python
app.save("output.hwp")
app.save("output.pdf")       # 포맷 자동 감지
app.save("output.docx")

app.close()
app.open("output.hwp")

text = app.get_text()
# 전체 문서 범위로 읽으려면:
from hwpapi import constants as const
full = app.get_text(
    spos=const.ScanStartPosition.Document,
    epos=const.ScanEndPosition.Document,
)
```

## 8. 필드 · 구조

```python
# 현재 날짜 삽입
app.actions.InsertStringDateTime.run()

# 페이지 번호
app.actions.InsertPageNum.run()

# 책갈피
action = app.actions.Bookmark
action.pset.Name = "ch1"
action.run()

# 하이퍼링크
action = app.actions.InsertHyperlink
action.pset.Text = "GitHub"
action.pset.Command = "https://github.com/JunDamin/hwpapi"
action.run()

# 다단 레이아웃
action = app.actions.MultiColumn  # pset에 ColDef
```

## 9. 액션 · ParameterSet 탐색

```python
# 전체 액션 (704개)
all_actions = app.actions.list_actions()

# pset을 받는 액션만 (~670개)
with_pset = app.actions.list_actions(with_pset_only=True)

# 특정 액션의 ParameterSet 클래스
CS = app.actions.get_pset_class('CharShape')
print(len(CS._property_registry))  # 65개 속성

# 모든 ParameterSet 레지스트리
from hwpapi.parametersets import PARAMETERSET_REGISTRY
print(len(PARAMETERSET_REGISTRY))  # 286
```

## 10. Escape hatch — win32com 직접 접근

hwpapi가 다루지 않는 기능은 `app.api`로 언제든 내려갈 수 있습니다.

```python
# 원시 HwpObject
hwp = app.api
hwp.Run("BreakPara")
hwp.PointToHwpUnit(20)

# HParameterSet 직접 조작 (docs의 고전적 패턴)
hwp.HAction.GetDefault("CharShape", hwp.HParameterSet.HCharShape.HSet)
hwp.HParameterSet.HCharShape.Bold = 1
hwp.HAction.Execute("CharShape", hwp.HParameterSet.HCharShape.HSet)
```

## 다음 단계

- [App 기초](01_app_basics.html) — `App` 객체와 기본 개념
- [찾기·바꾸기 심화](02_find_and_replace.html)
- [Core API](../api/00_core.html) — 전체 레퍼런스